<a href="https://colab.research.google.com/github/lautarodibartolo-ae/t8002-representacion-de-datos/blob/main/clase-3-dependencias-y-normalizacion/03_dependencias_y_normalizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Clase 3 — Los ejemplos

**Taller T8002 · Representación de datos**

Este notebook acompaña la clase en vivo, en seis bloques.

La clase 2 llegó al diseño de seis tablas **a ojo**. Hoy llegamos al mismo lugar por el camino
formal, y comprobamos que coinciden.

## Cómo se usa

**No hay nada que completar.** Ejecutá de arriba hacia abajo con `Shift + Enter` y leé cada
salida.

### Empezá por esta

In [ ]:
import pandas as pd
from itertools import combinations
from pathlib import Path

Path("crudo").mkdir(exist_ok=True)


# Dos ayudantes, para que las celdas de cada bloque muestren el concepto y no
# la fontaneria de pandas. Los dos hacen lo mismo que hariamos a mano.
def tabla_propia(origen, columnas, nombres, con_clave=True):
    # Saca esas columnas a una tabla nueva sin repetidos. Con con_clave, le
    # agrega una clave sustituta 1, 2, 3...
    nueva = origen[columnas].drop_duplicates().reset_index(drop=True).rename(columns=nombres)
    if con_clave:
        nueva.insert(0, "id", range(1, len(nueva) + 1))
    return nueva


def referenciar(origen, columna, destino, columna_destino, nombre_fk, quitar=()):
    # Reemplaza `columna` por una clave foránea que apunta a `destino`, y quita
    # las columnas que se fueron con la tabla nueva.
    puente = destino[["id", columna_destino]].rename(columns={"id": nombre_fk})
    return (origen.merge(puente, left_on=columna, right_on=columna_destino)
                  .drop(columns=[columna, columna_destino, *quitar])
                  .sort_values("id").reset_index(drop=True))


print("pandas", pd.__version__)

In [ ]:
tabla = """id_hogar,localidad,provincia,fecha_visita,encuestador_legajo,encuestador_nombre,encuestador_telefono,personas,ingreso,cobertura_salud,satisfaccion,servicios
1,Ramallo,Buenos Aires,2025-03-14,E04,"Sosa, Diego",3407-419854,3,185000,si,buena,agua;luz;gas
1,Ramallo,Buenos Aires,2025-06-12,E04,"Sosa, Diego",3407-419854,4,201000,si,buena,agua;luz;gas
2,Córdoba,Córdoba,2025-03-15,E02,"Ledesma, Julio",351-4778120,5,240500,no,regular,agua;luz
3,Concepción,Tucumán,2025-03-15,E03,"Bianchi, Ana",381-4551907,2,,,,luz
4,Ramallo,Buenos Aires,2025-03-16,E01,"Rivas, Marta",3407-412233,4,198000,si,buena,agua;luz;gas
5,Ramallo,Buenos Aires,2025-03-16,E04,"Sosa, Diego",3407-419854,1,120000,,,luz
7,Córdoba,Córdoba,2025-03-17,E04,"Sosa, Diego",3407-419854,6,310000,no,mala,agua;luz;gas;cloacas
7,Córdoba,Córdoba,2025-06-13,E04,"Sosa, Diego",3407-419854,6,325000,no,regular,agua;luz;gas;cloacas
8,Concepción,Tucumán,2025-03-17,E03,"Bianchi, Ana",381-4551907,3,175000,si,regular,agua;luz
9,Ramallo,Buenos Aires,2025-03-18,E04,"Sosa, Diego",3407-419854,2,,no,mala,luz;gas
10,Córdoba,Córdoba,2025-03-19,E02,"Ledesma, Julio",351-4778120,4,205000,,,agua;luz;gas
11,Concepción,Tucumán,2025-03-20,E03,"Bianchi, Ana",381-4551907,7,260000,si,buena,agua;luz;gas;cloacas
11,Concepción,Tucumán,2025-06-13,E03,"Bianchi, Ana",381-4551907,7,268000,si,buena,agua;luz;gas;cloacas
13,Ramallo,Buenos Aires,2025-03-20,E04,"Sosa, Diego",3407-419854,3,190000,,,agua;luz
14,Córdoba,Córdoba,2025-03-21,E02,"Ledesma, Julio",351-4778120,2,168000,no,regular,agua
15,Ramallo,Buenos Aires,2025-03-21,E04,"Sosa, Diego",3407-419854,3,172000,si,regular,agua;luz;gas
16,Concepción,Tucumán,2025-03-24,E03,"Bianchi, Ana",381-4551907,4,195000,no,buena,agua;luz
17,Córdoba,Córdoba,2025-03-24,E04,"Sosa, Diego",3407-419854,2,210000,si,regular,agua;luz;gas
18,Córdoba,Córdoba,2025-03-25,E02,"Ledesma, Julio",351-4778120,5,155000,,,luz
19,Ramallo,Buenos Aires,2025-03-25,E04,"Sosa, Diego",3407-419854,3,188000,no,mala,agua;luz
20,Concepción,Tucumán,2025-03-26,E03,"Bianchi, Ana",381-4551907,6,225000,si,buena,agua;luz;gas;cloacas
20,Concepción,Tucumán,2025-06-16,E03,"Bianchi, Ana",381-4551907,5,231000,si,regular,agua;luz;gas;cloacas
21,Ramallo,Buenos Aires,2025-03-26,E04,"Sosa, Diego",3407-419854,2,163000,,,agua;luz
22,Córdoba,Córdoba,2025-03-27,E02,"Ledesma, Julio",351-4778120,4,201000,si,regular,agua;luz;gas
23,Ramallo,Buenos Aires,2025-03-27,E04,"Sosa, Diego",3407-419854,3,179000,no,buena,agua;luz
24,Concepción,Tucumán,2025-03-28,E03,"Bianchi, Ana",381-4551907,5,232000,si,regular,agua;luz;gas
25,Córdoba,Córdoba,2025-03-28,E04,"Sosa, Diego",3407-419854,1,148000,,,luz
26,Ramallo,Buenos Aires,2025-03-31,E01,"Rivas, Marta",3407-412233,4,216000,no,regular,agua;luz;gas
26,Ramallo,Buenos Aires,2025-06-17,E01,"Rivas, Marta",3407-412233,4,222000,no,buena,agua;luz;gas
27,Concepción,Tucumán,2025-03-31,E03,"Bianchi, Ana",381-4551907,2,193000,si,mala,agua;luz
30,Ramallo,Buenos Aires,2025-04-01,E01,"Rivas, Marta",3407-412233,7,244000,si,buena,agua;luz;gas;cloacas
30,Ramallo,Buenos Aires,2025-06-18,E01,"Rivas, Marta",3407-412233,8,259000,si,buena,agua;luz;gas;cloacas
31,Concepción,Tucumán,2025-04-02,E03,"Bianchi, Ana",381-4551907,4,181000,,,agua;luz
32,Córdoba,Córdoba,2025-04-02,E02,"Ledesma, Julio",351-4778120,3,207000,no,regular,agua;luz;gas
33,Ramallo,Buenos Aires,2025-04-02,E04,"Sosa, Diego",3407-419854,5,169000,si,regular,agua;luz
34,Concepción,Tucumán,2025-04-03,E03,"Bianchi, Ana",381-4551907,2,236000,no,buena,agua;luz;gas
35,Córdoba,Córdoba,2025-04-03,E04,"Sosa, Diego",3407-419854,4,152000,,,agua
36,Ramallo,Buenos Aires,2025-04-04,E01,"Rivas, Marta",3407-412233,6,199000,si,mala,agua;luz;gas
36,Ramallo,Buenos Aires,2025-06-19,E01,"Rivas, Marta",3407-412233,6,204000,si,regular,agua;luz;gas
37,Concepción,Tucumán,2025-04-04,E03,"Bianchi, Ana",381-4551907,3,223000,no,regular,agua;luz
38,Córdoba,Córdoba,2025-04-07,E02,"Ledesma, Julio",351-4778120,2,0,si,mala,luz
39,Ramallo,Buenos Aires,2025-04-07,E04,"Sosa, Diego",3407-419854,12,620000,no,buena,agua;luz;gas;cloacas
39,Ramallo,Buenos Aires,2025-06-20,E04,"Sosa, Diego",3407-419854,12,641000,no,regular,agua;luz;gas;cloacas
40,Concepción,Tucumán,2025-04-08,E03,"Bianchi, Ana",381-4551907,4,186000,,,agua;luz
42,Ramallo,Buenos Aires,2025-04-09,E01,"Rivas, Marta",3407-412233,2,176000,no,mala,agua;luz
"""

Path("crudo/relevamiento_plano.csv").write_text(tabla, encoding="utf-8")
df = pd.read_csv("crudo/relevamiento_plano.csv")

CLAVE = ["id_hogar", "fecha_visita"]
print(f"{len(df)} filas, {len(df.columns)} columnas, {df['id_hogar'].nunique()} hogares")
print(f"clave: {' + '.join(CLAVE)}")
df[df["id_hogar"] == 1]

Las dos visitas del hogar 1. `personas` cambió de 3 a 4. `localidad` y `provincia` no cambiaron,
y no podían cambiar.

**Esa diferencia entre dos columnas de la misma tabla es todo el contenido de la clase.**

---
# 1. Buscar dependencias funcionales

Apunte, sección 2.3. El procedimiento: agrupar por A y contar cuántos valores distintos de B hay en
cada grupo. Si todos los grupos tienen exactamente uno, entonces A → B.

In [ ]:
def determina(datos, izquierda, derecha):
    """¿izquierda -> derecha? Devuelve (si_o_no, peor_cantidad_de_valores)."""
    izquierda = izquierda if isinstance(izquierda, list) else [izquierda]
    conteo = datos.groupby(izquierda, dropna=False)[derecha].nunique(dropna=False)
    return bool((conteo <= 1).all()), int(conteo.max())

# El ejemplo del apunte, paso a paso.
print("localidad -> provincia")
print(df.groupby("localidad")["provincia"].agg(["size", "nunique"])
        .rename(columns={"size": "filas", "nunique": "provincias distintas"}))
print()
print("localidad -> personas")
print(df.groupby("localidad")["personas"].agg(["size", "nunique"])
        .rename(columns={"size": "filas", "nunique": "personas distintas"}))

En la primera tabla, todos los grupos tienen **una** provincia. Entonces `localidad → provincia`.

En la segunda, los grupos tienen seis, seis y nueve valores distintos. Entonces `localidad` no
determina `personas`.

Es la misma prueba en los dos casos: *dado este valor, ¿el otro queda fijado?*

In [ ]:
# Ahora todos los pares de columnas, de una sola vez.
hallazgos = []
for a, b in combinations(df.columns, 2):
    for izq, der in [(a, b), (b, a)]:
        si, _ = determina(df, izq, der)
        if si:
            hallazgos.append((izq, der))

print(f"{len(hallazgos)} dependencias candidatas entre columnas sueltas\n")
for izq, der in hallazgos:
    print(f"   {izq}  ->  {der}")

Ahí están todas las candidatas de una columna a otra. Algunas son las que buscábamos, y varias
son basura: aparecen porque 45 filas son pocas. Eso es el bloque 2.

Falta un tipo que este barrido no encuentra: las que salen de la clave compuesta, donde el lado
izquierdo son **dos** columnas.

In [ ]:
# Las dependencias de la clave entera, y las de media clave.
print("DE LA CLAVE ENTERA (totales)")
for col in df.columns:
    if col in CLAVE:
        continue
    si, _ = determina(df, CLAVE, col)
    if si:
        print(f"   id_hogar, fecha_visita  ->  {col}")

print()
print("DE MEDIA CLAVE (parciales)")
for col in df.columns:
    if col in CLAVE:
        continue
    si, _ = determina(df, "id_hogar", col)
    if si:
        print(f"   id_hogar  ->  {col}")

La clave entera determina las diez columnas restantes, como corresponde.

En la segunda lista aparecen siete, y **solo dos son dependencias parciales**: `localidad` y
`servicios`, que son las dos que lista el apunte en su sección 2.6. `provincia` también aparece,
pero no es parcial: es transitiva a través de `localidad`.

Las otras cuatro —`encuestador_legajo`, `encuestador_nombre`, `encuestador_telefono` y
`cobertura_salud`— son casualidades de estas 45 filas: ningún hogar de este archivo cambió de
encuestador entre sus dos visitas, y eso no es una regla del operativo. El bloque 2 las descarta.

La cadena completa de `provincia` es `id_hogar → localidad → provincia`, y la sección 4.4 la
resuelve como transitiva.

---
# 2. Las candidatas que son casualidad

Apunte, sección 2.4. **El procedimiento no encuentra dependencias funcionales: encuentra
candidatas.**

In [ ]:
# Caso 1: provincia -> localidad pasa la prueba.
si, _ = determina(df, "provincia", "localidad")
print(f"¿provincia -> localidad pasa la prueba?  {si}")
print()
print(df.groupby("provincia")["localidad"].agg(["size", "nunique", "unique"]))

Pasa la prueba, y **es falso**.

Cada provincia tiene una sola localidad en este archivo, así que el procedimiento dice que sí. La
provincia de Buenos Aires tiene miles de localidades: el día que el relevamiento incorpore
Pergamino, la dependencia se cae.

Lo que pasó es que este relevamiento tiene una localidad por provincia. Es una casualidad de la
muestra, no una regla.

In [ ]:
# Caso 2: encuestador_legajo -> localidad. Un solo encuestador la salva.
resumen = df.groupby("encuestador_legajo")["localidad"].agg(["size", "nunique", "unique"])
print(resumen)
print()
si, peor = determina(df, "encuestador_legajo", "localidad")
print(f"¿pasa la prueba? {si}   (el peor grupo tiene {peor} localidades)")

E01, E02 y E03 trabajaron en una localidad cada uno. **E04 trabajó en dos, y eso mata la
dependencia.**

Si E04 no existiera, el procedimiento habría dicho que el encuestador determina la localidad. Y
seguiría siendo falso, porque nada en el operativo impide que un encuestador cubra dos localidades.

> Los datos **descartan** dependencias: un contraejemplo alcanza. Lo que no pueden hacer es
> **confirmarlas**. Una dependencia funcional es una regla del negocio, y se confirma preguntando.

In [ ]:
# Las diez dependencias que sobreviven a la pregunta por la regla.
reales = {
    "totales": [(CLAVE, c) for c in ["encuestador_legajo", "personas", "ingreso",
                                     "cobertura_salud", "satisfaccion"]],
    "parciales": [(["id_hogar"], "localidad"), (["id_hogar"], "servicios")],
    "transitivas": [(["localidad"], "provincia"),
                    (["encuestador_legajo"], "encuestador_nombre"),
                    (["encuestador_legajo"], "encuestador_telefono")],
}
for forma, lista in reales.items():
    print(forma.upper())
    for izq, der in lista:
        print(f"   {' , '.join(izq):24} ->  {der}")
    print()

Las cinco totales están bien: es lo que se espera de una tabla.

Las cinco que no son totales **son la redundancia, escrita en notación**. Tres de ellas, como
muestra:

| Dependencia | Por eso se repite |
|---|---|
| `id_hogar → localidad` | la localidad del hogar 1 está dos veces |
| `localidad → provincia` | `Ramallo — Buenos Aires` está veinte veces |
| `encuestador_legajo → encuestador_telefono` | el teléfono de E04 está diecisiete veces |

Eso es lo que hace útil todo este aparato: convierte "acá hay algo repetido" en una **lista finita**
de cosas concretas que se arreglan de a una.

---
# 3. Una descomposición mala

Apunte, sección 3.3. Separar es fácil; separar bien, no.

In [ ]:
# Tres hogares distintos, los mismos del apunte: 1 y 4 en Ramallo, 2 en Cordoba.
chico = (df[df["id_hogar"].isin([1, 2, 4])]
         .drop_duplicates("id_hogar")[["id_hogar", "localidad", "personas"]]
         .reset_index(drop=True))
print("PARTIMOS DE:")
print(chico.to_string(index=False))

# Buena: id_hogar queda en las dos tablas.
buena_a = chico[["id_hogar", "localidad"]]
buena_b = chico[["id_hogar", "personas"]]
vuelta_buena = buena_a.merge(buena_b, on="id_hogar")

print("\nDESCOMPOSICIÓN BUENA (id_hogar en las dos)")
print(vuelta_buena.to_string(index=False))
print(f"filas: {len(vuelta_buena)}  ->  ¿idéntica al original? "
      f"{vuelta_buena.equals(chico)}")

Tres filas, idénticas. Nada se perdió.

In [ ]:
# Mala: sin ninguna columna en comun.
mala_a = chico[["id_hogar", "personas"]]
mala_b = chico[["localidad"]].drop_duplicates()
vuelta_mala = mala_a.merge(mala_b, how="cross")   # sin columna comun, todo con todo

print("DESCOMPOSICIÓN MALA (sin columna compartida)")
print(vuelta_mala.to_string(index=False))
print(f"\nfilas: {len(vuelta_mala)} donde había {len(chico)}")
print(f"filas inventadas: {len(vuelta_mala) - len(chico)}")

Seis filas donde había tres, y **tres son inventadas**: dicen que el hogar 1 está en Córdoba, y que
los hogares 2 y 4 están cada uno en la localidad del otro.

Y lo peor es que no parecen inventadas. Tienen la misma forma que las verdaderas.

> **La regla:** al separar una tabla en dos, la columna que las une tiene que quedar en las dos. Es
> exactamente la regla 2 del pasaje de la clase 2.

---
# 4. Los tres pasos

Apunte, sección 4.6. De la tabla plana a seis tablas, un paso por forma normal.

### Paso 1 → primera forma normal: un valor por celda

In [ ]:
multivalor = df["servicios"].str.contains(";").sum()
print(f"filas con más de un valor en la celda: {multivalor} de {len(df)}")
print(f"ejemplo: {df.loc[0, 'servicios']!r}")
print()

servicio = pd.DataFrame({"id": [1, 2, 3, 4], "nombre": ["agua", "luz", "gas", "cloacas"]})
hogar_servicio = (df[["id_hogar", "servicios"]].drop_duplicates()
    .assign(servicios=lambda d: d["servicios"].str.split(";"))
    .explode("servicios")
    .merge(servicio, left_on="servicios", right_on="nombre")
    [["id_hogar", "id"]]
    .rename(columns={"id_hogar": "hogar_id", "id": "servicio_id"})
    .drop_duplicates().sort_values(["hogar_id", "servicio_id"]).reset_index(drop=True))

r1 = df.drop(columns=["servicios"])
print(f"servicio:       {len(servicio)} filas")
print(f"hogar_servicio: {len(hogar_servicio)} filas")
print(f"queda:          {len(r1)} filas x {len(r1.columns)} columnas")
print("\nTABLAS: 3")

La celda con `agua;luz;gas` se convirtió en tres filas de `hogar_servicio`.

> **Cómo se reconoce una violación de 1FN:** celdas con separadores adentro, y columnas con nombre
> en plural. `servicios` tenía las dos señales.

### Paso 2 → segunda forma normal: nada depende de una parte de la clave

In [ ]:
# Las dependencias parciales que quedan en r1.
print("dependencias parciales pendientes:")
for col in r1.columns:
    if col in CLAVE:
        continue
    si, _ = determina(r1, "id_hogar", col)
    if si:
        print(f"   id_hogar -> {col}")
print()

# Lo que depende solo de id_hogar se va a una tabla propia.
hogar = (r1[["id_hogar", "localidad", "provincia"]].drop_duplicates()
         .rename(columns={"id_hogar": "id"}).sort_values("id").reset_index(drop=True))
visita = (r1.drop(columns=["localidad", "provincia"])
          .rename(columns={"id_hogar": "hogar_id", "fecha_visita": "fecha"})
          .sort_values(["hogar_id", "fecha"]).reset_index(drop=True))

print(f"hogar:  {len(hogar)} filas x {len(hogar.columns)} columnas")
print(f"visita: {len(visita)} filas x {len(visita.columns)} columnas")
print("\nTABLAS: 4")
hogar.head(4)

La localidad del hogar 1 quedó escrita **una vez**, no dos.

La salida de arriba lista seis parciales pendientes y el código mueve dos. Las otras cuatro son las
casualidades del bloque 1: si las moviéramos, estaríamos separando por una regla que nadie
confirmó.

Y la tabla `visita` ya no tiene `localidad` ni `provincia`. No se fueron por comodidad: se fueron
porque **nunca fueron datos de la visita**.

### Paso 3 → tercera forma normal: nada depende de un atributo que no es clave

In [ ]:
# Primero el diagnostico: que transitivas quedan, y en que tabla.
print("transitivas en hogar:")
for col in hogar.columns:
    if col != "localidad" and determina(hogar, "localidad", col)[0] and col != "id":
        print(f"   localidad -> {col}")
print("transitivas en visita:")
for col in visita.columns:
    if col.startswith("encuestador_") and col != "encuestador_legajo":
        if determina(visita, "encuestador_legajo", col)[0]:
            print(f"   encuestador_legajo -> {col}")

Tres cadenas: una en `hogar` y dos en `visita`. Ahora el arreglo.

In [ ]:
# El atributo del medio de la cadena se lleva a una tabla propia.
localidad = tabla_propia(hogar, ["localidad", "provincia"], {"localidad": "nombre"})
hogar_3fn = referenciar(hogar, "localidad", localidad, "nombre", "localidad_id",
                        quitar=["provincia"])

encuestador = tabla_propia(
    visita, ["encuestador_legajo", "encuestador_nombre", "encuestador_telefono"],
    {"encuestador_legajo": "legajo", "encuestador_nombre": "nombre",
     "encuestador_telefono": "telefono"}, con_clave=False)
visita_3fn = visita.drop(columns=["encuestador_nombre", "encuestador_telefono"])

print(f"localidad:   {len(localidad)} filas")
print(f"encuestador: {len(encuestador)} filas")
print("\nTABLAS: 6")

Ahora el teléfono de E04 está en **una** celda.

La anomalía de modificación de la clase 1 no se mitigó: **dejó de ser posible.** No hay diecisiete
lugares que puedan discrepar, porque hay uno.

---
# 5. La verificación contra la clase 2

Esta es la razón de ser de la clase. La clase 2 llegó a seis tablas mirando qué cosas existen. Hoy
llegamos a seis tablas mirando qué dependencias hay. **Tienen que ser las mismas.**

In [ ]:
# Izquierda: lo que salio hoy. Derecha: el esquema del apunte de la clase 2, seccion 4.4.
hoy = {"localidad": localidad, "encuestador": encuestador, "hogar": hogar_3fn,
       "visita": visita_3fn, "servicio": servicio, "hogar_servicio": hogar_servicio}
clase_2 = {
    "localidad":      ["id", "nombre", "provincia"],
    "encuestador":    ["legajo", "nombre", "telefono"],
    "hogar":          ["id", "localidad_id"],
    "visita":         ["hogar_id", "fecha", "encuestador_legajo", "personas", "ingreso",
                       "cobertura_salud", "satisfaccion"],
    "servicio":       ["id", "nombre"],
    "hogar_servicio": ["hogar_id", "servicio_id"],
}

for nombre, esperadas in clase_2.items():
    coincide = sorted(hoy[nombre].columns) == sorted(esperadas)
    print(f"   {nombre:16} {'coincide' if coincide else 'NO COINCIDE'}")

print("\n¿Los dos caminos dan el mismo esquema?",
      all(sorted(hoy[n].columns) == sorted(c) for n, c in clase_2.items()))

Las seis coinciden, tabla por tabla y columna por columna.

Eso no es una comprobación decorativa: es la señal de que los dos métodos describen la misma cosa.

> **El diseño conceptual y la normalización no son alternativas: son un método y su verificación.**
> Se diseña conceptualmente, porque es la forma de hablar con quien conoce el negocio. Se verifica
> con las dependencias, porque es la forma de saber si quedó algo repetido adentro.

---
# 6. El cruce final, y la vista

Apunte, secciones 4.6 y 5.2. La última prueba: **¿se recupera la tabla original?**

In [ ]:
# Se cruzan las cuatro tablas que tienen las columnas de la tabla plana.
vista = (visita_3fn
    .merge(hogar_3fn.rename(columns={"id": "hogar_id"}), on="hogar_id")
    .merge(localidad.rename(columns={"id": "localidad_id", "nombre": "localidad"}),
           on="localidad_id")
    .merge(encuestador.rename(columns={"legajo": "encuestador_legajo",
                                       "nombre": "encuestador_nombre",
                                       "telefono": "encuestador_telefono"}),
           on="encuestador_legajo")
    .rename(columns={"hogar_id": "id_hogar", "fecha": "fecha_visita"}))

print(f"{len(vista)} filas, {len(vista.columns)} columnas")

Falta `servicios`, que vive en `hogar_servicio` y se vuelve a armar aparte.

In [ ]:
servicios_texto = (hogar_servicio
    .merge(servicio, left_on="servicio_id", right_on="id")
    .sort_values(["hogar_id", "servicio_id"])
    .groupby("hogar_id")["nombre"].apply(";".join).rename("servicios"))

vista = vista.merge(servicios_texto, left_on="id_hogar", right_index=True)

izq = vista[list(df.columns)].sort_values(CLAVE).reset_index(drop=True)
der = df.sort_values(CLAVE).reset_index(drop=True)

print(f"la vista tiene {len(izq)} filas x {len(izq.columns)} columnas")
print(f"el original tiene {len(der)} filas x {len(der.columns)} columnas")
print()
print("¿Son idénticas, celda por celda?", izq.equals(der))

`True`. Las 45 filas y las 12 columnas, valor por valor.

Eso es una **vista**: una consulta guardada con nombre que se usa como si fuera una tabla. Y la
propiedad que la hace valiosa:

In [ ]:
# Sobre una copia, para que volver a ejecutar el bloque anterior siga dando lo mismo.
encuestador_corregido = encuestador.copy()
encuestador_corregido.loc[encuestador_corregido["legajo"] == "E04", "telefono"] = "3407-500100"

# Y volvemos a armar la vista, sin tocar nada mas.
vista_nueva = (visita_3fn
    .merge(encuestador_corregido.rename(columns={"legajo": "encuestador_legajo",
                                                 "nombre": "encuestador_nombre",
                                                 "telefono": "encuestador_telefono"}),
           on="encuestador_legajo"))

print("celdas que hubo que cambiar:", 1)
print("filas de la vista que salen corregidas:",
      (vista_nueva["encuestador_telefono"] == "3407-500100").sum())
print()
print(vista_nueva["encuestador_telefono"].value_counts().to_string())

**Una celda cambiada, diecisiete filas corregidas.** Y no hay ninguna forma de que quede a medias,
porque hay un solo lugar donde escribir.

Compará con el bloque 3 del notebook de la clase 1, donde el mismo cambio dejó dos teléfonos para la
misma persona.

Y notá lo que la vista **no** hace: no guarda los datos. Cada vez que se la consulta, cruza las
tablas de nuevo. Así que se ve la tabla plana completa y el dato sigue estando escrito una sola vez.
Es la comodidad sin la redundancia.

---
# Cierre

| Bloque | Lo que quedó |
|---|---|
| 1 | Dieciséis candidatas entre columnas sueltas, y siete de media clave |
| 2 | `provincia → localidad` pasa la prueba y es falsa |
| 3 | Sin columna compartida, el cruce inventa filas que parecen datos |
| 4 | 1FN, 2FN y 3FN: de una tabla a seis, un defecto por paso |
| 5 | Los dos caminos dan el mismo esquema |
| 6 | La vista devuelve las 45 filas, y un cambio es una celda |

**Y la advertencia final**, que es la que más se olvida: las dependencias que encontrás en los datos
son candidatas. Antes de separar una tabla, confirmá la regla con quien conoce el negocio. Separar
por una casualidad de la muestra deja un diseño que se rompe con la fila siguiente.

Eso es el trabajo final.